[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module7/01-neural-networks.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module7/01-neural-networks.ipynb)

# Neural Networks from Scratch
**Module 7 — Lesson 1 | Estimated time: 45 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Understand how a single perceptron computes its output
- Implement and visualise step, sigmoid, ReLU, tanh, and GELU activations
- Perform a manual forward pass through a 2-layer network
- Compute MSE and cross-entropy loss
- Derive backpropagation step-by-step using the chain rule
- Compare manually computed gradients against `torch.autograd`
- Visualise the decision boundary learned by a 2-layer network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 1. The Perceptron

A perceptron is the simplest neural network unit. It computes a weighted sum of its inputs, adds a bias, and passes the result through an activation function:

```
output = activation(w · x + b)
```

where `w` is the weight vector, `x` is the input vector, and `b` is the scalar bias.

In [ ]:
class Perceptron:
    """Single perceptron implemented from scratch with NumPy."""
    def __init__(self, n_inputs, activation='step'):
        self.weights = np.random.randn(n_inputs) * 0.1
        self.bias = 0.0
        self.activation = activation

    def _activate(self, z):
        if self.activation == 'step':
            return (z >= 0).astype(float)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        elif self.activation == 'relu':
            return np.maximum(0, z)
        raise ValueError(f'Unknown activation: {self.activation}')

    def forward(self, x):
        z = np.dot(x, self.weights) + self.bias
        return self._activate(z)

    def fit(self, X, y, lr=0.1, epochs=20):
        """Perceptron learning rule."""
        history = []
        for _ in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                pred = self.forward(xi)
                delta = yi - pred
                self.weights += lr * delta * xi
                self.bias += lr * delta
                errors += int(delta != 0)
            history.append(errors)
        return history

# Train on AND gate
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)

p = Perceptron(n_inputs=2, activation='step')
history = p.fit(X_and, y_and, lr=0.1, epochs=50)
predictions = [p.forward(x) for x in X_and]
print('AND gate predictions:', predictions)
print('Expected:           ', y_and.tolist())
plt.plot(history)
plt.title('Perceptron Training Errors per Epoch')
plt.xlabel('Epoch'); plt.ylabel('Errors')
plt.tight_layout(); plt.show()

## 2. Activation Functions

Activation functions introduce non-linearity, enabling networks to learn complex mappings.

| Function | Formula | Use case |
|----------|---------|----------|
| Step | `1 if z≥0 else 0` | Original perceptron |
| Sigmoid | `1/(1+e^-z)` | Binary output |
| ReLU | `max(0, z)` | Hidden layers (default) |
| Tanh | `(e^z - e^-z)/(e^z + e^-z)` | RNNs, zero-centred |
| GELU | `z·Φ(z)` | Transformers (BERT, GPT) |

In [ ]:
z = np.linspace(-5, 5, 300)

def step(z):    return (z >= 0).astype(float)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)
def tanh(z):    return np.tanh(z)
def gelu(z):    return z * 0.5 * (1 + np.vectorize(lambda x: __import__('math').erf(x / 2**0.5))(z))

functions = {
    'Step': (step, 'steelblue'),
    'Sigmoid': (sigmoid, 'darkorange'),
    'ReLU': (relu, 'green'),
    'Tanh': (tanh, 'red'),
    'GELU': (gelu, 'purple'),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, (name, (fn, color)) in zip(axes, functions.items()):
    ax.plot(z, fn(z), color=color, lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.axvline(0, color='k', lw=0.5, ls='--')
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('z'); ax.set_ylim(-1.5, 1.5)
plt.suptitle('Activation Functions', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

## 3. Manual Forward Pass Through a 2-Layer Network

A 2-layer (one hidden layer) network with architecture `input(2) → hidden(4) → output(1)`:

```
Layer 1:  Z1 = X @ W1 + b1,   A1 = relu(Z1)
Layer 2:  Z2 = A1 @ W2 + b2,  A2 = sigmoid(Z2)   # output probability
```

In [ ]:
# Network dimensions
n_in, n_h, n_out = 2, 4, 1

# He initialisation for ReLU layers
W1 = np.random.randn(n_in, n_h) * np.sqrt(2 / n_in)
b1 = np.zeros((1, n_h))
W2 = np.random.randn(n_h, n_out) * np.sqrt(2 / n_h)
b2 = np.zeros((1, n_out))

# Sample input (batch of 5)
X = np.random.randn(5, n_in)

# Forward pass
Z1 = X @ W1 + b1
A1 = np.maximum(0, Z1)          # ReLU
Z2 = A1 @ W2 + b2
A2 = 1 / (1 + np.exp(-Z2))     # Sigmoid

print('Input shape:         ', X.shape)
print('Z1 shape:            ', Z1.shape)
print('A1 (after ReLU):     ', A1.shape)
print('Z2 shape:            ', Z2.shape)
print('A2 (output probs):   ', A2.shape)
print('\nSample outputs (first 3):'); print(A2[:3].round(4))

## 4. Loss Functions

**Mean Squared Error (MSE)** — regression:
$$\mathcal{L}_{MSE} = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

**Binary Cross-Entropy** — binary classification:
$$\mathcal{L}_{BCE} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

In [ ]:
y_true = np.array([[1],[0],[1],[1],[0]], dtype=float)
y_pred = A2  # sigmoid outputs

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def bce(y_true, y_pred, eps=1e-8):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

print(f'MSE loss:            {mse(y_true, y_pred):.4f}')
print(f'BCE loss:            {bce(y_true, y_pred):.4f}')

# Visualise BCE landscape for a single prediction
p_range = np.linspace(0.01, 0.99, 200)
loss_y1 = -np.log(p_range)       # true label = 1
loss_y0 = -np.log(1 - p_range)   # true label = 0

plt.figure(figsize=(7, 3))
plt.plot(p_range, loss_y1, label='y=1 (−log p̂)', color='blue')
plt.plot(p_range, loss_y0, label='y=0 (−log(1−p̂))', color='red')
plt.xlabel('Predicted probability p̂'); plt.ylabel('Loss')
plt.title('Binary Cross-Entropy Loss Landscape')
plt.legend(); plt.tight_layout(); plt.show()

## 5. Backpropagation — Step by Step

Backprop applies the **chain rule** to compute `∂L/∂W` for every layer, flowing gradients from output back to input.

```
Output layer:
  dL/dZ2 = A2 - y          (BCE + sigmoid derivative simplifies nicely)
  dL/dW2 = A1ᵀ @ dL/dZ2
  dL/db2 = sum(dL/dZ2)

Hidden layer:
  dL/dA1 = dL/dZ2 @ W2ᵀ
  dL/dZ1 = dL/dA1 * relu'(Z1)   where relu'(z) = 1 if z>0 else 0
  dL/dW1 = Xᵀ @ dL/dZ1
  dL/db1 = sum(dL/dZ1)
```

In [ ]:
m = X.shape[0]  # batch size

# Output layer gradient (BCE + sigmoid)
dZ2 = A2 - y_true                    # (m, 1)
dW2 = (A1.T @ dZ2) / m              # (n_h, n_out)
db2 = np.sum(dZ2, axis=0, keepdims=True) / m

# Hidden layer gradient
dA1 = dZ2 @ W2.T                     # (m, n_h)
dZ1 = dA1 * (Z1 > 0).astype(float)  # ReLU derivative
dW1 = (X.T @ dZ1) / m               # (n_in, n_h)
db1 = np.sum(dZ1, axis=0, keepdims=True) / m

print('Manual gradients:')
print(f'  dW1 shape: {dW1.shape}, dW2 shape: {dW2.shape}')
print(f'  dW1 sample: {dW1[0].round(5)}')
print(f'  dW2 sample: {dW2[:,0].round(5)}')

## 6. Manual Gradients vs torch.autograd

PyTorch's autograd engine computes the exact same gradients automatically — let's verify they match.

In [ ]:
# Convert everything to PyTorch tensors
X_t  = torch.tensor(X,      dtype=torch.float32, requires_grad=False)
y_t  = torch.tensor(y_true, dtype=torch.float32)
W1_t = torch.tensor(W1,     dtype=torch.float32, requires_grad=True)
b1_t = torch.tensor(b1,     dtype=torch.float32, requires_grad=True)
W2_t = torch.tensor(W2,     dtype=torch.float32, requires_grad=True)
b2_t = torch.tensor(b2,     dtype=torch.float32, requires_grad=True)

# Forward pass (mirrors NumPy version)
Z1_t = X_t @ W1_t + b1_t
A1_t = torch.relu(Z1_t)
Z2_t = A1_t @ W2_t + b2_t
A2_t = torch.sigmoid(Z2_t)

# Loss
loss_t = nn.BCELoss()(A2_t, y_t)
loss_t.backward()

print('Gradient comparison (W1[0]):')
print(f'  NumPy manual:   {dW1[0].round(6)}')
print(f'  torch.autograd: {W1_t.grad[0].numpy().round(6)}')

close = np.allclose(dW1, W1_t.grad.numpy(), atol=1e-5)
print(f'\nGradients match: {close}')

## 7. Decision Boundary Visualisation

Train a 2-layer network with PyTorch on a moons dataset and visualise its learned decision boundary.

In [ ]:
from sklearn.datasets import make_moons

# Generate data
X_m, y_m = make_moons(n_samples=400, noise=0.2, random_state=42)
X_t2 = torch.tensor(X_m, dtype=torch.float32)
y_t2 = torch.tensor(y_m, dtype=torch.float32).unsqueeze(1)

# Model
model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid()
)
optimiser = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

# Training loop
losses = []
for epoch in range(500):
    model.train()
    pred = model(X_t2)
    loss = criterion(pred, y_t2)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    if epoch % 50 == 0:
        losses.append(loss.item())

# Plot decision boundary
xx, yy = np.meshgrid(np.linspace(-2.5, 3.5, 200), np.linspace(-1.5, 2.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    Z = model(grid).numpy().reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
cmap = ListedColormap(['#FFAAAA', '#AAAAFF'])
ax1.contourf(xx, yy, Z, levels=[0, 0.5, 1], cmap=cmap, alpha=0.6)
ax1.scatter(X_m[:,0], X_m[:,1], c=y_m, cmap='bwr', edgecolors='k', s=20)
ax1.set_title('Learned Decision Boundary'); ax1.set_xlabel('x1'); ax1.set_ylabel('x2')
ax2.plot(losses)
ax2.set_title('Training Loss'); ax2.set_xlabel('Epoch × 50'); ax2.set_ylabel('BCE Loss')
plt.tight_layout(); plt.show()
acc = ((model(X_t2) > 0.5).float() == y_t2).float().mean()
print(f'Training accuracy: {acc.item():.3f}')

## Practice Exercises

**Exercise 1 — XOR Perceptron**
A single perceptron cannot learn the XOR function. Verify this empirically: train a single `Perceptron` on XOR data and confirm it fails to converge. Then add a hidden layer (manually, using NumPy) and show XOR can be solved.

**Exercise 2 — Custom Activation**
Implement the **Swish** activation function (`z * sigmoid(z)`) in NumPy, plot it alongside ReLU and GELU, and replace ReLU in the 2-layer network above. Compare training loss curves.

**Exercise 3 — Gradient Check**
Implement numerical gradient checking using the finite-difference formula `(L(w+ε) - L(w-ε)) / (2ε)` for `ε=1e-5`. Apply it to verify your manual backprop gradients match the numerical approximation to within `1e-4`.